In [5]:
import pandas as pd


try:
    df = pd.read_excel('Sample - Superstore.xls', engine='xlrd')
except FileNotFoundError:
    print("Error: 'Sample - Superstore.xls' not found in the directory.")
    
except ImportError:
    print("Error: Please install 'xlrd' to read .xls files (pip install xlrd)")


df['Order Date'] = pd.to_datetime(df['Order Date'])


df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['YearMonth'] = df['Order Date'].dt.to_period('M')


print('=== BASIC STATS ===')
print(f'Total Revenue: ${df["Sales"].sum():,.0f}')
print(f'Total Profit: ${df["Profit"].sum():,.0f}')

sales_sum = df["Sales"].sum()
profit_sum = df["Profit"].sum()
margin = (profit_sum / sales_sum * 100) if sales_sum != 0 else 0
print(f'Profit Margin: {margin:.1f}%')
print(f'Total Orders: {df["Order ID"].nunique()}')
print(f'Total Customers: {df["Customer ID"].nunique()}')
print(f'Date Range: {df["Order Date"].min().date()} to {df["Order Date"].max().date()}')


print()
print('=== YEARLY REVENUE ===')
print(df.groupby('Year')[['Sales','Profit']].sum().to_string())

print()
print('=== BY CATEGORY ===')
cat = df.groupby('Category').agg({'Sales':'sum','Profit':'sum','Order ID':'count'}).rename(columns={'Order ID':'Orders'})
cat['Margin%'] = cat['Profit']/cat['Sales']*100
print(cat.to_string())

print()
print('=== BY SUB-CATEGORY ===')
sub = df.groupby('Sub-Category').agg({'Sales':'sum','Profit':'sum'})
sub['Margin%'] = sub['Profit']/sub['Sales']*100
print(sub.sort_values('Sales', ascending=False).to_string())

print()
print('=== BY REGION ===')
reg = df.groupby('Region').agg({'Sales':'sum','Profit':'sum','Order ID':'count'}).rename(columns={'Order ID':'Orders'})
reg['Margin%'] = reg['Profit']/reg['Sales']*100
print(reg.sort_values('Sales', ascending=False).to_string())

print()
print('=== TOP 10 PRODUCTS ===')
prod = df.groupby('Product Name').agg({'Sales':'sum','Profit':'sum'})
prod['Margin%'] = prod['Profit']/prod['Sales']*100
print(prod.sort_values('Sales', ascending=False).head(10).to_string())

print()
print('=== BY SEGMENT ===')
seg = df.groupby('Segment').agg({'Sales':'sum','Profit':'sum','Customer ID':'nunique'}).rename(columns={'Customer ID':'Customers'})
seg['Margin%'] = seg['Profit']/seg['Sales']*100
print(seg.to_string())

print()
print('=== MONTHLY TREND (yearly totals) ===')
monthly = df.groupby(['Year', df['Order Date'].dt.month])['Sales'].sum().unstack()
print(monthly.to_string())


=== BASIC STATS ===
Total Revenue: $2,297,201
Total Profit: $286,397
Profit Margin: 12.5%
Total Orders: 5009
Total Customers: 793
Date Range: 2014-01-03 to 2017-12-30

=== YEARLY REVENUE ===
            Sales      Profit
Year                         
2014  484247.4981  49543.9741
2015  470532.5090  61618.6037
2016  609205.5980  81795.1743
2017  733215.2552  93439.2696

=== BY CATEGORY ===
                       Sales       Profit  Orders    Margin%
Category                                                    
Furniture        741999.7953   18451.2728    2121   2.486695
Office Supplies  719047.0320  122490.8008    6026  17.035158
Technology       836154.0330  145454.9481    1847  17.395712

=== BY SUB-CATEGORY ===
                    Sales      Profit    Margin%
Sub-Category                                    
Phones        330007.0540  44515.7306  13.489327
Chairs        328449.1030  26590.1663   8.095673
Storage       223843.6080  21278.8264   9.506113
Tables        206965.5320 -17725.